In [ ]:
from datasets import Dataset, load_dataset
from transformers import AutoModelForVision2Seq, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
from tqdm.auto import tqdm 
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer
from transformers import Qwen2VLProcessor

import pandas as pd 
import torch
import time

In [ ]:
df = pd.read_csv("final_df.csv")

In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
hf_dataset = Dataset.from_pandas(df)

In [ ]:
system_message = "당신은 입력 이미지가 'AI 생성 이미지'인지 '사람이 촬영한 실제 사진'인지 판별하는 분류 모델입니다."
prompt = """입력으로 이미지가 주어지면, 해당 이미지가 AI가 생성한 이미지인지 여부를 "0" 또는 "1"로 출력하세요.
- 출력은 반드시 숫자 하나만(문자열/설명 없이) 출력합니다.
  - "0" : AI 생성 이미지(생성형 모델로 만들어졌거나 합성/렌더링/딥페이크 등 인공적으로 생성된 경우 포함)
  - "1" : 실제 사진(사람이 카메라로 촬영한 사진)

판별 시 아래 단서를 종합적으로 고려하세요(단, 어느 한 가지 단서만으로 단정하지 마세요):
1. **질감/디테일의 비현실성**
   - 피부, 머리카락, 천/금속/유리의 질감이 과도하게 매끈하거나 반복 패턴이 보임
   - 미세 디테일(문자, 로고, 손가락, 치아, 눈동자, 귀 등)이 어색하거나 깨짐
2. **조명/그림자/반사의 불일치**
   - 광원 방향과 그림자 방향이 맞지 않음
   - 반사(거울/유리/물)에서 왜곡된 물체 형태 또는 논리적으로 불가능한 반사
3. **기하학/구조의 오류**
   - 배경의 선(창틀, 건물, 도로, 가구)이 비정상적으로 휘거나 이어짐이 부자연스러움
   - 물체 경계가 흐리거나 주변과 섞이는 현상
4. **사진적 특성**
   - 카메라 촬영에서 흔한 노이즈/입자감/심도(아웃포커싱)/모션블러가 자연스럽게 나타나는지
   - 과도하게 선명하거나, 반대로 특정 영역만 비정상적으로 뭉개지는지
5. **전반적 일관성**
   - 장면의 의미/맥락이 자연스러운지, 비현실적 구성 요소가 섞여 있는지
6. **워터마크/메타데이터 흔적**
   - AI 생성기(Stable Diffusion, Midjourney, DALL-E, Flux 등)의 시그니처 패턴: 
     - Stable Diffusion: 픽셀에서 반복되는 미세 노이즈나 주기적 패턴 (seed 기반)
     - Midjourney: 독특한 "V" 모양 아티팩트, 과도한 대칭성
     - DALL-E/OpenAI: 보이지 않는 SynthID 같은 고주파 노이즈나 비자연스러운 텍스처 그레인
     - Flux/Grok 등 신형 모델: 과도하게 깨끗한 이미지나 비현실적 선명도
     - Nano Banana (Gemini): 바나나 아이콘, Gemini sparkle(별 모양), SynthID 노이즈 패턴
     - 보이지 않는 워터마크 흔적: 전체적으로 비자연스러운 노이즈 분포, 픽셀값의 비랜덤성, 고주파 영역 이상
7. **로고/시각적 워터마크**
   - "AI generated", "Midjourney", "DALL-E" 등의 텍스트/로고가 미세하게 남아 있거나, 배경에 블렌딩됨

위 기준을 바탕으로 AI 생성으로 판단되면 "0", 실제 사진으로 판단되면 "1"을 출력하십시오.
출력은 오직 숫자 하나(1 또는 0)만 반환하고, 어떠한 추가 문구나 설명도 첨부하지 마십시오.
반드시 출력물에는 숫자 하나(1 또는 0)가 존재해야합니다.
"""

In [ ]:
train = hf_dataset.filter(lambda x : x['split'] == "train")
test = hf_dataset.filter(lambda x : x['split'] == "test")

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
train[0]

{'file_id': 'ai_000353',
 'file_path': '/workspace/ai-vs-real/train/ai/000353.jpg',
 'split': 'train',
 'binary_label': 0,
 'is_real': False}

In [ ]:
# 데이터셋을 OpenAI 메시지 형식으로 변환      
def format_data(sample):
   return {"messages": [
               {
                   "role": "system",
                   "content": [{"type": "text", "text": system_message}],
               },
               {
                   "role": "user",
                   "content": [
                       {"type" : "text", "text" : prompt},
                       {
                           "type": "image",
                           "image": sample["file_path"],
                       }
                   ],
               },
               {
                   "role": "assistant",
                   "content": [{"type": "text", "text": sample["binary_label"]}],
               },
           ],
       }


train_dataset = [format_data(sample) for sample in train]
test_dataset = [format_data(sample) for sample in test]

In [ ]:
print("학습 데이터 :", len(train_dataset))
print("테스트 데이터 :", len(test_dataset))

학습 데이터 : 1600
테스트 데이터 : 400


In [ ]:
model_id = "Qwen/Qwen2-VL-7B-Instruct" 

model = AutoModelForVision2Seq.from_pretrained(
   model_id,
   device_map="auto", 
   torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [ ]:
text = processor.apply_chat_template(
    train_dataset[246]["messages"], tokenize=False, add_generation_prompt=False
)
print(text)

<|im_start|>system
당신은 입력 이미지가 'AI 생성 이미지'인지 '사람이 촬영한 실제 사진'인지 판별하는 분류 모델입니다.<|im_end|>
<|im_start|>user
입력으로 이미지가 주어지면, 해당 이미지가 AI가 생성한 이미지인지 여부를 "0" 또는 "1"로 출력하세요.
- 출력은 반드시 숫자 하나만(문자열/설명 없이) 출력합니다.
  - "0" : AI 생성 이미지(생성형 모델로 만들어졌거나 합성/렌더링/딥페이크 등 인공적으로 생성된 경우 포함)
  - "1" : 실제 사진(사람이 카메라로 촬영한 사진)

판별 시 아래 단서를 종합적으로 고려하세요(단, 어느 한 가지 단서만으로 단정하지 마세요):
1. **질감/디테일의 비현실성**
   - 피부, 머리카락, 천/금속/유리의 질감이 과도하게 매끈하거나 반복 패턴이 보임
   - 미세 디테일(문자, 로고, 손가락, 치아, 눈동자, 귀 등)이 어색하거나 깨짐
2. **조명/그림자/반사의 불일치**
   - 광원 방향과 그림자 방향이 맞지 않음
   - 반사(거울/유리/물)에서 왜곡된 물체 형태 또는 논리적으로 불가능한 반사
3. **기하학/구조의 오류**
   - 배경의 선(창틀, 건물, 도로, 가구)이 비정상적으로 휘거나 이어짐이 부자연스러움
   - 물체 경계가 흐리거나 주변과 섞이는 현상
4. **사진적 특성**
   - 카메라 촬영에서 흔한 노이즈/입자감/심도(아웃포커싱)/모션블러가 자연스럽게 나타나는지
   - 과도하게 선명하거나, 반대로 특정 영역만 비정상적으로 뭉개지는지
5. **전반적 일관성**
   - 장면의 의미/맥락이 자연스러운지, 비현실적 구성 요소가 섞여 있는지
6. **워터마크/메타데이터 흔적**
   - AI 생성기(Stable Diffusion, Midjourney, DALL-E, Flux 등)의 시그니처 패턴: 
     - Stable Diffusion: 픽셀에서 반복되는 미세 노이즈나 주기적 패턴 (seed 기반)
     - Midjourney: 독특한

In [ ]:
MAX_SIDE = 1024

# 이미지 리사이징
def _resize_to_max_side(img: Image.Image, max_side: int) -> Image.Image:
    w, h = img.size
    scale = max_side / max(w, h)
    if scale >= 1:
        return img
    new_size = (int(w * scale), int(h * scale))
    return img.resize(new_size, Image.BICUBIC)

def to_pil(p):
    if isinstance(p, str):
        return Image.open(p).convert("RGB")
    if isinstance(p, Image.Image):
        return p.convert("RGB")
    return p

    
# 모델 답변 생성
def generate_ai_or_real(messages, model, processor):
   text = processor.apply_chat_template(
       messages, tokenize=False, add_generation_prompt=True
   )
   image_inputs, video_inputs = process_vision_info(messages)

   resized_images = []
   for p in image_inputs:
        img = to_pil(p)
        img = _resize_to_max_side(img, MAX_SIDE)
        resized_images.append(img)

    
   inputs = processor(
       text=[text],
       images=resized_images,
       videos=video_inputs,
       padding=True,
       return_tensors="pt",
   )
   inputs = inputs.to(model.device)
   
   generated_ids = model.generate(
      **inputs,
      max_new_tokens=20,
      top_p=0.95,
      do_sample=True,
      temperature = 0.05
      )
   
   generated_ids_trimmed = [
      out_ids[len(in_ids) :] 
      for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

   output_text = processor.batch_decode(
       generated_ids_trimmed, 
       skip_special_tokens=True, 
       clean_up_tokenization_spaces=False
   )
   return output_text[0]

# 재시도
def retry_process(messages, model, processor, sleep_sec=0.0, max_tries=10):
    tries = 0
    while True:
        pred = generate_ai_or_real(messages, model, processor).strip()
        if pred != "":
            return pred

        tries += 1
        if max_tries is not None and tries >= max_tries:
            return "" 

        if sleep_sec > 0:
            time.sleep(sleep_sec)

In [ ]:
messages =  train_dataset[246]["messages"]
answer = train_dataset[246]["messages"][2]["content"][0]["text"]
base_description = retry_process(messages, model, processor, sleep_sec=0.0, max_tries=10)
print("예측 결과 :", base_description)
print("정답 결과 :", answer)

예측 결과 : 1
정답 결과 : 0


In [ ]:
# 학습 전 예
no_train_result = [] 

for idx in tqdm(range(len(test_dataset))):
    messages =  test_dataset[idx]["messages"]
    answer = test_dataset[idx]["messages"][2]["content"][0]["text"]
    base_description = retry_process(messages, model, processor, sleep_sec=0.0, max_tries=10)
    no_train_result.append((answer, base_description))

  0%|          | 0/400 [00:00<?, ?it/s]

In [ ]:
prediction_result = [int(predict) == int(answer) for predict, answer in no_train_result]
before_train_accuracy = sum(prediction_result) /  len(prediction_result) * 100
print(f"학습 전 정확도 : {before_train_accuracy:.2f}%") 

학습 전 정확도 : 70.25%


In [ ]:
# LoRA 설정
peft_config = LoraConfig(
        lora_alpha=16,
        lora_dropout=0.05,
        r=8,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM"
)

In [ ]:
# SFTConfig 학습설정
args = SFTConfig(
    output_dir="qwen2-7b-instruct-ai-generated-detector-4000",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=5,
    save_strategy="epoch",
    learning_rate=2e-4,
    bf16=True,
    tf32=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="wandb", 
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True}
)


args.remove_unused_columns = False

In [ ]:
MAX_SIDE = 768 # 1024로 학습시 OOM문제가 발생하여 768로 설정

def collate_fn(examples):
    texts = [processor.apply_chat_template(example["messages"], tokenize=False) for example in examples]
    images = []
    for ex in examples:
        img_inputs, vid_inputs = process_vision_info(ex["messages"])
       
        x = img_inputs[0] if isinstance(img_inputs, list) else img_inputs
        img = to_pil(x)
        img = _resize_to_max_side(img, MAX_SIDE)
        images.append(img)

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    if isinstance(processor, Qwen2VLProcessor):  
        image_tokens = [151652, 151653, 151655]
    else:
        image_tokens = [processor.tokenizer.convert_tokens_to_ids(processor.image_token)]

    for image_token_id in image_tokens:
        labels[labels == image_token_id] = -100
    
    batch["labels"] = labels

    return batch

In [ ]:
# 단일 예시 확인
example = test_dataset[1]
print("단일 예시 데이터:")
print(example)

단일 예시 데이터:
{'messages': [{'role': 'system', 'content': [{'type': 'text', 'text': "당신은 입력 이미지가 'AI 생성 이미지'인지 '사람이 촬영한 실제 사진'인지 판별하는 분류 모델입니다."}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '입력으로 이미지가 주어지면, 해당 이미지가 AI가 생성한 이미지인지 여부를 "0" 또는 "1"로 출력하세요.\n- 출력은 반드시 숫자 하나만(문자열/설명 없이) 출력합니다.\n  - "0" : AI 생성 이미지(생성형 모델로 만들어졌거나 합성/렌더링/딥페이크 등 인공적으로 생성된 경우 포함)\n  - "1" : 실제 사진(사람이 카메라로 촬영한 사진)\n\n판별 시 아래 단서를 종합적으로 고려하세요(단, 어느 한 가지 단서만으로 단정하지 마세요):\n1. **질감/디테일의 비현실성**\n   - 피부, 머리카락, 천/금속/유리의 질감이 과도하게 매끈하거나 반복 패턴이 보임\n   - 미세 디테일(문자, 로고, 손가락, 치아, 눈동자, 귀 등)이 어색하거나 깨짐\n2. **조명/그림자/반사의 불일치**\n   - 광원 방향과 그림자 방향이 맞지 않음\n   - 반사(거울/유리/물)에서 왜곡된 물체 형태 또는 논리적으로 불가능한 반사\n3. **기하학/구조의 오류**\n   - 배경의 선(창틀, 건물, 도로, 가구)이 비정상적으로 휘거나 이어짐이 부자연스러움\n   - 물체 경계가 흐리거나 주변과 섞이는 현상\n4. **사진적 특성**\n   - 카메라 촬영에서 흔한 노이즈/입자감/심도(아웃포커싱)/모션블러가 자연스럽게 나타나는지\n   - 과도하게 선명하거나, 반대로 특정 영역만 비정상적으로 뭉개지는지\n5. **전반적 일관성**\n   - 장면의 의미/맥락이 자연스러운지, 비현실적 구성 요소가 섞여 있는지\n6. **워터마크/메타데이터 흔적**\n   - AI 생성기(Stable Diffusion, Midj

In [ ]:
# collate_fn 테스트 (배치 크기 1로)
batch = collate_fn([example])
print("\n처리된 배치 데이터:")
print("입력 ID 형태:", batch["input_ids"].shape)
print("어텐션 마스크 형태:", batch["attention_mask"].shape)
print("이미지 픽셀 형태:", batch["pixel_values"].shape)
print("레이블 형태:", batch["labels"].shape)


처리된 배치 데이터:
입력 ID 형태: torch.Size([1, 1685])
어텐션 마스크 형태: torch.Size([1, 1685])
이미지 픽셀 형태: torch.Size([2916, 1176])
레이블 형태: torch.Size([1, 1685])


In [ ]:
batch

{'input_ids': tensor([[151644,   8948,    198,  ...,     15, 151645,    198]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]]), 'pixel_values': tensor([[ 1.8135,  1.8135,  1.8135,  ...,  0.1409,  0.1551,  0.1551],
        [ 1.7990,  1.7990,  1.7990,  ...,  0.1266,  0.1266,  0.1266],
        [ 1.7844,  1.7844,  1.7990,  ...,  0.1409,  0.1409,  0.1409],
        ...,
        [-1.6317, -1.6317, -1.6463,  ..., -0.9541, -0.9399, -0.9256],
        [-1.7047, -1.7047, -1.7193,  ..., -1.0394, -1.0110, -0.9399],
        [-1.1791, -1.2083, -1.2083,  ..., -0.9114, -0.9114, -0.9114]]), 'image_grid_thw': tensor([[ 1, 54, 54]]), 'labels': tensor([[151644,   8948,    198,  ...,     15, 151645,    198]])}

In [ ]:
print('입력에 대한 정수 인코딩 결과:')
print(batch["input_ids"][0])

입력에 대한 정수 인코딩 결과:
tensor([151644,   8948,    198,  ...,     15, 151645,    198])


In [ ]:
print('레이블에 대한 정수 인코딩 결과:')
print(batch["labels"][0])

레이블에 대한 정수 인코딩 결과:
tensor([151644,   8948,    198,  ...,     15, 151645,    198])


In [ ]:
decoded_text = processor.tokenizer.decode(batch["input_ids"][0])
print("\n디코딩된 텍스트:")
print(decoded_text)


디코딩된 텍스트:
<|im_start|>system
당신은 입력 이미지가 'AI 생성 이미지'인지 '사람이 촬영한 실제 사진'인지 판별하는 분류 모델입니다.<|im_end|>
<|im_start|>user
입력으로 이미지가 주어지면, 해당 이미지가 AI가 생성한 이미지인지 여부를 "0" 또는 "1"로 출력하세요.
- 출력은 반드시 숫자 하나만(문자열/설명 없이) 출력합니다.
  - "0" : AI 생성 이미지(생성형 모델로 만들어졌거나 합성/렌더링/딥페이크 등 인공적으로 생성된 경우 포함)
  - "1" : 실제 사진(사람이 카메라로 촬영한 사진)

판별 시 아래 단서를 종합적으로 고려하세요(단, 어느 한 가지 단서만으로 단정하지 마세요):
1. **질감/디테일의 비현실성**
   - 피부, 머리카락, 천/금속/유리의 질감이 과도하게 매끈하거나 반복 패턴이 보임
   - 미세 디테일(문자, 로고, 손가락, 치아, 눈동자, 귀 등)이 어색하거나 깨짐
2. **조명/그림자/반사의 불일치**
   - 광원 방향과 그림자 방향이 맞지 않음
   - 반사(거울/유리/물)에서 왜곡된 물체 형태 또는 논리적으로 불가능한 반사
3. **기하학/구조의 오류**
   - 배경의 선(창틀, 건물, 도로, 가구)이 비정상적으로 휘거나 이어짐이 부자연스러움
   - 물체 경계가 흐리거나 주변과 섞이는 현상
4. **사진적 특성**
   - 카메라 촬영에서 흔한 노이즈/입자감/심도(아웃포커싱)/모션블러가 자연스럽게 나타나는지
   - 과도하게 선명하거나, 반대로 특정 영역만 비정상적으로 뭉개지는지
5. **전반적 일관성**
   - 장면의 의미/맥락이 자연스러운지, 비현실적 구성 요소가 섞여 있는지
6. **워터마크/메타데이터 흔적**
   - AI 생성기(Stable Diffusion, Midjourney, DALL-E, Flux 등)의 시그니처 패턴: 
     - Stable Diffusion: 픽셀에서 반복되는 미세 노이즈나 주기적 패턴 (seed 기반)
     - Midj

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    dataset_text_field="",      
    peft_config=peft_config,
    tokenizer=processor.tokenizer,
)

In [ ]:
trainer.train()

Step,Training Loss
5,2.079100
10,1.920800
15,1.677900
20,1.375100
25,1.037900


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


TrainOutput(global_step=25, training_loss=1.6181472969055175, metrics={'train_runtime': 738.8554, 'train_samples_per_second': 2.166, 'train_steps_per_second': 0.034, 'total_flos': 1.25387042906112e+17, 'train_loss': 1.6181472969055175, 'epoch': 1.0})

In [ ]:
# 메모리 해제
del model
del trainer
torch.cuda.empty_cache()

In [ ]:
model_id = "Qwen/Qwen2-VL-7B-Instruct" 

model = AutoModelForVision2Seq.from_pretrained(
  model_id,
  device_map="auto",
  torch_dtype=torch.bfloat16
)
processor = AutoProcessor.from_pretrained(model_id)

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# 로라 어댑터가 있는 경로
adapter_path1 = "./qwen2-7b-instruct-ai-generated-detector-4000/checkpoint-25"

# 첫 번째 Adapter 로드
model.load_adapter(
    adapter_path1,
    adapter_name="adapter1")

In [ ]:
test_result = [] 

model.set_adapter("adapter1")

# 평가 
for idx in tqdm(range(len(test_dataset))):
    messages =  test_dataset[idx]["messages"]
    answer = test_dataset[idx]["messages"][2]["content"][0]["text"]
    ft_real_image = retry_process(messages, model, processor, max_tries=10)
    test_result.append((int(answer), int(ft_real_image)))

after_prediction_result = [int(predict) == int(answer) for predict, answer in test_result]
after_train_accuracy = sum(after_prediction_result) /  len(after_prediction_result) * 100
print(f"학습 후 정확도 : {after_train_accuracy:.2f}%") 

  0%|          | 0/400 [00:00<?, ?it/s]

학습 후 정확도 : 91.50%


In [ ]:
print(f"학습 전 정확도 : {before_train_accuracy:.2f}%") 
print(f"학습 후 정확도 : {after_train_accuracy:.2f}%")

학습 전 정확도 : 70.25%
학습 후 정확도 : 91.50%


In [ ]:
# 모델 병합 후 저장
adapter_path = "./qwen2-7b-instruct-ai-generated-detector-4000/checkpoint-25"
base_model_id = "Qwen/Qwen2-VL-7B-Instruct" 
merged_path = "qwen2-7b-instruct-ai-generated-detector-4000-checkpoint-25"

model = AutoModelForVision2Seq.from_pretrained(base_model_id, low_cpu_mem_usage=True)

peft_model = PeftModel.from_pretrained(model, adapter_path)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(merged_path, safe_serialization=True, max_shard_size="5GB")

processor = AutoProcessor.from_pretrained(base_model_id)
processor.save_pretrained(merged_path)

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

[]

- 모델 업로드

In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="./env")

True

In [ ]:
import os
from huggingface_hub import HfApi

api_key = os.environ.get("HUGGINGFACE_KEY")
api = HfApi()

username = "eddyfox8812"
MODEL_NAME = "qwen2-7b-instruct-ai-generated-detector-4000-checkpoint-25"

api.create_repo(
    token=api_key,
    repo_id=f"{username}/{MODEL_NAME}",
    repo_type="model"
)

api.upload_folder(
    token=api_key,
    repo_id=f"{username}/{MODEL_NAME}",
    folder_path="qwen2-7b-instruct-ai-generated-detector-4000-checkpoint-25",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/eddyfox8812/qwen2-7b-instruct-ai-generated-detector-4000-checkpoint-25/commit/f832198648f22ee7e44a0bf6c5165998309c938d', commit_message='Upload folder using huggingface_hub', commit_description='', oid='f832198648f22ee7e44a0bf6c5165998309c938d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/eddyfox8812/qwen2-7b-instruct-ai-generated-detector-4000-checkpoint-25', endpoint='https://huggingface.co', repo_type='model', repo_id='eddyfox8812/qwen2-7b-instruct-ai-generated-detector-4000-checkpoint-25'), pr_revision=None, pr_num=None)